## Loading preprocessed delta file and creating dim_product

In [0]:
# Importing libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, monotonically_increasing_id

In [0]:
#Initializing spark session
spark = SparkSession.builder.appName("Preprocessor").getOrCreate()

In [0]:
# File location and type
file_location = "/FileStore/tables/Fact_Sales_Preprocessed"
file_type = "delta"

# The applied options are for delta files. For other file types, these will be ignored.
df_product = spark.read.format(file_type) \
  .option("inferSchema", True) \
  .option("header", True) \
  .load(file_location)


In [0]:

df_product_final = df_product.select("ProductCategory", "ProductName", "Brand").distinct()

In [0]:
#Skipping default value and creating surrogate key as we have already handled null values

 #as monotonically_increasing_id() starts with 0, we do monotonically_increasing_id()+1
df_product_final = df_product_final.withColumn("Product_Id", monotonically_increasing_id()+1) \
    .select("Product_Id","ProductCategory", "ProductName", "Brand")


In [0]:
df_product_final.display()

Product_Id,ProductCategory,ProductName,Brand
1,Electronics,Smartphone,BrandC
2,Electronics,Smartphone,BrandB
3,Electronics,T-shirt,BrandA
4,Electronics,Desk,BrandC
5,Clothing,Tablet,BrandA
6,Electronics,Chair,BrandA
7,Furniture,Desk,BrandA
8,Furniture,Tablet,BrandA
9,Electronics,Desk,BrandA
10,Electronics,Tablet,BrandB


In [0]:
df_product_final.write \
    .format("delta") \
        .mode("overwrite")\
            .save("/FileStore/tables/DIM_Product")